# ASTRA

本 notebook 合并了原 00/02/03: 环境自检 -> 输入卡 (表单或文本) ->
运行 ASTRA -> 输出清单。与官方 astra 程序对应。

In [1]:
%run _bootstrap.py

astra-notebook 后端已加载 (v0.1.0)
项目根目录: /Users/yuxinwu/my_projects/astra_notebook
模拟工作目录: /Users/yuxinwu/my_projects/astra_notebook/data/workspace
ASTRA    : /Users/yuxinwu/programs/ASTRA/astra
Generator: /Users/yuxinwu/programs/ASTRA/generator


In [2]:
# 环境自检
import numpy, scipy, matplotlib, pandas
print("numpy", numpy.__version__, "| scipy", scipy.__version__,
      "| matplotlib", matplotlib.__version__, "| pandas", pandas.__version__)
print()
print("ASTRA    :", ASTRA_EXE)
print("Generator:", GENERATOR_EXE)
print("工作目录 :", SIM_DIR)
print("下一步: 打开 01_generator.ipynb 生成初始束团。")

numpy 2.5.2 | scipy 1.18.0 | matplotlib 3.11.1 | pandas 3.0.5

ASTRA    : /Users/yuxinwu/programs/ASTRA/astra
Generator: /Users/yuxinwu/programs/ASTRA/generator
工作目录 : /Users/yuxinwu/my_projects/astra_notebook/data/workspace
下一步: 打开 01_generator.ipynb 生成初始束团。


## 输入卡 — 表单模式

按 namelist 分组, 覆盖手册第 6 章全部 13 个 namelist; 基础组已填入
可运行的默认值。

In [3]:
from astra_tools.widgets.forms import namelist_form
from astra_tools.deck.metadata import summary
print(summary())
print()
print('用法: namelist_form("NEWRUN", only=[...])  # 常用参数')
print('      namelist_form("NEWRUN")              # 全部参数')

  NEWRUN      (6.1) 38 参数
  OUTPUT      (6.2) 33 参数
  SCAN        (6.3) 18 参数
  MODULES     (6.4) 12 参数
  ERROR       (6.5) 57 参数
  CHARGE      (6.6) 27 参数
  APERTURE    (6.7) 21 参数
  WAKE        (6.8) 22 参数
  CAVITY      (6.9) 46 参数
  SOLENOID    (6.10) 12 参数
  QUADRUPOLE  (6.11) 16 参数
  DIPOLE      (6.13) 11 参数
  LASER       (6.14) 26 参数
  INPUT       (7) 66 参数

用法: namelist_form("NEWRUN", only=[...])  # 常用参数
      namelist_form("NEWRUN")              # 全部参数


In [ ]:
# 常用追踪参数表单 (逐组生成)
forms, getters = {}, {}
# 基础组: 以可运行的默认值作种子 (纯漂移, 产生全部输出文件)
_basic_seed = {
    "NEWRUN": {"Track_All": True, "Auto_Phase": True,
               "check_ref_part": False, "H_max": 0.001, "H_min": 0.0,
               "Xoff": 0.0, "Yoff": 0.0},
    "OUTPUT": {"ZSTART": 0.0, "ZSTOP": 1.5, "Zemit": 100, "Zphase": 1,
               "RefS": True, "EmitS": True, "PhaseS": True, "SigmaS": True},
}
groups = {
    "NEWRUN": ["Head", "RUN", "Distribution", "Track_All", "Auto_Phase",
               "check_ref_part", "H_max", "H_min", "Xoff", "Yoff", "Toff"],
    "OUTPUT": ["ZSTART", "ZSTOP", "Zemit", "Zphase", "RefS", "EmitS", "PhaseS",
               "SigmaS", "C_EmitS", "Lsub_cor", "Binary", "High_res"],
    "CHARGE": ["LSPCH", "Nrad", "Cell_var", "Nlong_in", "min_grid", "Max_Scale"],
    "CAVITY": ["LEfield", "File_Efield", "C_pos", "Nue", "MaxE", "Phi"],
    "SOLENOID": ["LBfield", "File_Bfield", "S_pos", "MaxB", "S_higher_order"],
    "WAKE": ["LWAKE", "Wk_filename", "Wk_z"],
    "APERTURE": ["LAPERT", "File_Aperture", "Ap_R"],
}
for gname, only in groups.items():
    wmap, getter = namelist_form(gname, values=_basic_seed.get(gname, {}), only=only)
    forms[gname] = wmap
    getters[gname] = getter
print("各组表单已生成 (基础组已填入可运行默认值)。")

In [5]:
# 写 astra.in (表单模式)
from astra_tools.namelist.write import write_input_deck

blocks = {}
for gname, getter in getters.items():
    # 基础组 (NEWRUN/OUTPUT) 全量写入; 其余组只写用户改动过的参数
    vals = getter(changed_only=(gname not in _basic_seed))
    if vals:
        blocks[gname] = vals
blocks.setdefault("NEWRUN", {})["Distribution"] = "'bunch.ini'"  # 相对路径
write_input_deck(blocks, SIM_DIR / "astra.in",
                 header="generated by astra-notebook (02_astra)")
print("astra.in 已写入, 包含 namelist:", list(blocks))
print()
print((SIM_DIR / "astra.in").read_text())

astra.in 已写入, 包含 namelist: ['NEWRUN', 'OUTPUT']

! generated by astra-notebook (02_astra)
&NEWRUN
  RUN=1,
  Xoff=0,
  Yoff=0,
  Toff=0,
  Track_All=T,
  Auto_Phase=T,
  check_ref_part=F,
  H_max=0.001,
  H_min=0,
  Distribution='bunch.ini',
 /

&OUTPUT
  ZSTART=0,
  ZSTOP=1.5,
  Zemit=100,
  Zphase=1,
  Lsub_cor=F,
  RefS=T,
  EmitS=T,
  C_EmitS=F,
  PhaseS=T,
  High_res=F,
  Binary=F,
  SigmaS=T,
 /




**文本模式**: 把现成 .in 文件复制为 `data/workspace/astra.in` 即可,
跳过表单直接运行下方单元。

## 运行 ASTRA

运行 data/workspace/astra.in, 实时日志流式输出, 结束后列出输出文件。

In [6]:
from astra_tools.run import run_program, discover_outputs

input_path = SIM_DIR / "astra.in"
if not input_path.exists():
    raise FileNotFoundError("astra.in 不存在 — 请先运行上方设置单元")
run_program(ASTRA_EXE, SIM_DIR, input_file="astra.in", timeout=3600)
print("ASTRA 运行完成。")

 --------------------------------------------------------------------------

                     Astra- A space charge tracking algorithm     
               Version 4.0 - macOS 64bit - OpenMPI - Apple Silicon
                              DESY,  Hamburg 2022                 
                         18. 8.2026  15:18

     Parameter file is:  astra.in                                          
                                                                                     

 Initialize element settings:
     neglecting space charge forces 

 --------------------------------------------------------------------------
     500 particles from file bunch.ini                                         

     Particles taken into account      N =        500
     total charge                      Q =     -1.000     nC
     horizontal beam position          x =     2.9830E-04 mm
     vertical beam position            y =     1.6413E-04 mm
     longitudinal beam position        z =     8.5491

In [7]:
import pandas as pd
outs = discover_outputs(SIM_DIR, "astra", run="001")
rows = []
for key, val in outs.items():
    if isinstance(val, list):
        rows += [(key, str(f.name), f.stat().st_size) for f in val]
    elif val is not None:
        rows += [(key, str(val.name), val.stat().st_size)]
pd.DataFrame(rows, columns=["类型", "文件", "大小(字节)"])

,类型,文件,大小(字节)
0,emit,astra.Xemit.001,8500
1,yemit,astra.Yemit.001,8500
2,zemit,astra.Zemit.001,8500
3,sigma,astra.Sigma.001,39200
4,ref,astra.ref.001,545436
5,log,astra.Log.001,15455
6,phase,astra.0150.001,52500


下一步: `03_postpro.ipynb` (相空间与统计) / `04_lineplot.ipynb` (演化)。